# Syllabus-Grounded Contextual Biasing — full study on a Colab GPU

Runs the whole plan (§3–§13) on a GPU instead of a laptop CPU. On CPU int8 the measured
real-time factor is ~1.5, which makes the full matrix roughly a twenty-hour job. On a T4
with float16 it is typically 5–15× faster, so the same work is a couple of hours.

**Runtime → Change runtime type → T4 GPU** before running anything.

### Two rules that keep the results valid

1. **Every number in one results table must come from one platform.** GPU float16 and CPU
   int8 produce different hypotheses. The cache key includes `device` and `compute_type`,
   so the two can never silently mix inside a cache — but you must not put a CPU row and a
   GPU row in the same table. Run the whole matrix here.
2. **Re-run the §4.2 model pilot here.** It is cheap on a GPU, and on a GPU `large-v3` is
   affordable, which removes the "knowingly weakened baseline" criticism that using the
   turbo variant invites (§10, threat 1).

In [ ]:
#@title 1. Check the GPU
!nvidia-smi

In [ ]:
#@title 2. Get the code
# Requires the work to be pushed to GitHub first (git push origin main).
%cd /content
!git clone -q https://github.com/meet244/FYP.git || (cd FYP && git pull -q)
%cd /content/FYP
!ls src | head -40

In [ ]:
#@title 3. Dependencies
# requirements.txt pins CPU-oriented versions (numpy<2 etc.) that fight Colab's
# preinstalled torch, so install the minimum set instead and leave torch alone.
!pip install -q faster-whisper==1.1.1 "ctranslate2==4.5.0" jiwer==3.0.4 \
    rapidfuzz indic-transliteration sentence-transformers soundfile librosa \
    pyyaml matplotlib 2>&1 | tail -3
!python -c "import ctranslate2, faster_whisper; print('ctranslate2', ctranslate2.__version__)"

In [ ]:
#@title 4. Make cuDNN visible to CTranslate2
# CTranslate2 loads cuDNN from the NVIDIA pip wheels rather than from Colab's default
# library path. Without this you get: 'Unable to load libcudnn_ops.so.9'.
import os, glob, site, subprocess, sys
libdirs = set()
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    for p in glob.glob(os.path.join(sp, 'nvidia', '*', 'lib')):
        libdirs.add(p)
os.environ['LD_LIBRARY_PATH'] = ':'.join(sorted(libdirs)) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('\n'.join(sorted(libdirs)) or 'no nvidia wheel lib dirs found')

# A GPU decode must be proven to work *now*, not at hour three of the matrix.
smoke = '''
from faster_whisper import WhisperModel
import numpy as np, soundfile as sf
sf.write('/tmp/probe.wav', np.zeros(16000, dtype='float32'), 16000)
m = WhisperModel('tiny', device='cuda', compute_type='float16')
list(m.transcribe('/tmp/probe.wav', language='hi')[0])
print('GPU decode OK')
'''
print(subprocess.run([sys.executable, '-c', smoke], capture_output=True, text=True,
                     env=os.environ).stdout or 'FAILED — see stderr below')
# If it failed, run:  !pip install -q nvidia-cudnn-cu12 nvidia-cublas-cu12
# then re-run this cell.

In [ ]:
#@title 5. Point the frozen config at the GPU
# device, compute_type and size are part of the ASR cache key, so this starts a clean
# cache rather than mixing GPU results with any CPU decodes in the repo.
import re, pathlib
p = pathlib.Path('configs/config.yaml'); t = p.read_text()
t = re.sub(r'^(\s*size:).*$', r'\1 large-v3           # GPU makes the stronger model affordable (§4.2)', t, count=1, flags=re.M)
t = re.sub(r'^(\s*compute_type:).*$', r'\1 float16    # GPU precision', t, count=1, flags=re.M)
t = re.sub(r'^(\s*device:).*$', r'\1 cuda            # Colab T4', t, count=1, flags=re.M)
p.write_text(t)
!sed -n '/^model:/,/^decode:/p' configs/config.yaml

In [ ]:
#@title 6. Corpus: download, cut, refine boundaries, freeze tiers  (~15 min)
import os
os.environ['PYTHONPATH'] = 'src'
!mkdir -p data/raw/slr104
!cd data/raw/slr104 && curl -sL -C - -o Hindi-English_test.tar.gz \
    https://openslr.trmal.net/resources/104/Hindi-English_test.tar.gz && tar -xzf Hindi-English_test.tar.gz
!python src/prepare_slr104.py 2>&1 | tail -8
# The distributed segments are whole-second windows and unusable as-is; see
# report/01_dataset_and_harness.md §2. This applies the boundary refinement.
!python src/refine_segments.py --radius 2.5 --lam 2.0 2>&1 | tail -6
!python src/make_tiers.py --force
!python src/build_syllabus.py 2>&1 | tail -2

In [ ]:
#@title 7. Self-tests (seconds, no GPU) — never skip before a long run
!python src/selftest.py 2>&1 | tail -3
!python src/selftest_pipeline.py 2>&1 | tail -3

In [ ]:
#@title 8. Benchmark, then the pilots  (§9.3, §4.2, §4.3)
!python src/bench_inference.py --n 12 --configs 1x0 2>&1 | tail -6
!python src/pilots.py language --tier tier1 2>&1 | grep -Ev 'utt/s' | tail -8
!python src/pilots.py model --tier tier1 2>&1 | grep -Ev 'utt/s' | tail -8
!python src/apply_pilot_decisions.py

In [ ]:
#@title 9. Baseline gate, Tier-1 tuning, Tier-2 matrix
# Each stage gates the next (§11). Inspect the printed gate before moving on.
!python src/run_matrix.py baseline --tier tier1 2>&1 | grep -Ev 'utt/s' | tail -30

In [ ]:
!python src/run_matrix.py tune --tier tier1 2>&1 | grep -Ev 'utt/s' | tail -40

In [ ]:
!python src/run_matrix.py matrix --tier tier2 2>&1 | grep -Ev 'utt/s' | tail -60

In [ ]:
#@title 10. Tier 3: the final confirmation, baseline + best system only
# Set `best` from the Tier-2 table before running this.
best = 'G'  #@param ["G", "M2", "M1", "M2+M3a", "M3a"]
!python src/run_matrix.py final --tier tier3 --best {best} 2>&1 | grep -Ev 'utt/s' | tail -30

In [ ]:
#@title 11. Results
!python src/status.py
from IPython.display import Markdown, Image, display
import pathlib
for f in ['report/results_tier2.md', 'report/results_tier3.md']:
    if pathlib.Path(f).exists():
        display(Markdown(pathlib.Path(f).read_text()))
for f in sorted(pathlib.Path('report/figures').glob('*.png')):
    print(f); display(Image(str(f)))

In [ ]:
#@title 12. Save everything to Drive before the runtime disconnects
# Colab's disk is ephemeral. `runs/` and `report/` are the results; `cache/` is the
# expensive decode archive and is worth keeping so a re-run resumes instead of repeating.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/FYP_results
!tar -czf /content/drive/MyDrive/FYP_results/runs_report.tar.gz runs report configs data/manifests
!tar -czf /content/drive/MyDrive/FYP_results/asr_cache.tar.gz cache/asr
!ls -lh /content/drive/MyDrive/FYP_results/
# To resume in a later session, after cells 1-5:
#   !tar -xzf /content/drive/MyDrive/FYP_results/asr_cache.tar.gz
#   !tar -xzf /content/drive/MyDrive/FYP_results/runs_report.tar.gz